In [ ]:
import numpy as np
import pynq

### Load the bitfile onto the Programmable Logic (PL)

In [ ]:
overlay = pynq.Overlay('METAccelerator.bit')

### Get the reference to the `METAccelerator` IP

In [ ]:
met_accelerator = overlay.METAccelerator

### Get the reference to the `register_map` and print it

The register map is our main interface to interact with the IP by setting values to the registers and reading them

In [ ]:
reg = met_accelerator.register_map
reg

### Function to load the CSV files

Here we don't want the `awkward` format like in the reference implementation.
This function "unrolls" the array into a 1D array of _all_ the `pt` and `phi` values, and creates another array for the `n_particles` per event.

In [ ]:
def load_jagged_csv(filepath):
    flat_data = []
    counts = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
        
            row = [float(val) for val in line.split(',') if val.strip()]
        
            flat_data.extend(row)
            counts.append(len(row))
        
    flat_array = np.array(flat_data, dtype=np.float32)
    counts_array = np.array(counts, dtype=np.float32)
    
    return flat_array, counts_array

### Load the data with the function above

In [ ]:
pt, n_particles = load_jagged_csv('../MET/pt.csv')
phi, _ = load_jagged_csv('../MET/phi.csv')
n_events = len(n_particles)

### Allocate buffers

To send data to the pynq, we need to use `pynq`'s buffers, which we can memory-map between the PS and PL. In this step we create buffers with the correct size and data type. Then we write the `pt`, `phi`, and `n_particles` data into their respective buffers. Finally, we create a buffer for the met result.

In [ ]:
# allocate buffers for data
pt_buff = pynq.allocate(len(pt), dtype=np.float32)
phi_buff = pynq.allocate(len(phi), dtype=np.float32)
n_particles_buff = pynq.allocate(n_events, dtype=np.int32)

# write data into buffers
pt_buff[:] = pt
phi_buff[:] = phi
n_particles_buff[:] = n_particles

# allocate a buffer for the result
met_buff = pynq.allocate(n_events, dtype=np.float32)

### Map the buffers to the MET IP

`n_events` is our only scalar variable, so for that one we directly write the value. For the other, data, ports, we write the physical address of the allocated buffers to the register in the MET IP. When the MET IP reads data from the buffers, it will read from those physical addresses.

In [ ]:
met_accelerator.write(reg.n_events.address, n_events)
met_accelerator.write(reg.n_particles.address, n_particles_buff.physical_address)
met_accelerator.write(reg.pt.address, pt_buff.physical_address)
met_accelerator.write(reg.phi.address, phi_buff.physical_address)
met_accelerator.write(reg.met.address, met_buff.physical_address)

### Compute the MET!

To finally run the MET computation in the FPGA, we send a "start" signal to the MET IP by writing to the `AP_START` bit in the control registers.

In [ ]:
met_accelerator.write(reg.CTRL.AP_START, 1)

### Inspect the result

Print out the contents of the MET buffer, which should correspond to the same values you saw in the C Simulation

In [ ]:
met_buff